In [ ]:
!pip install -qU langchain langchain-groq langchain-community langchain-text-splitters langchain-huggingface chromadb pypdf sentence-transformers streamlit
!npm install -g localtunnel


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3

In [ ]:
import streamlit as st
import os

from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import TokenTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


# Load environment variables
load_dotenv()

# Secure API Key
GROQ_API_KEY = os.getenv("os.getenv("GROQ_API_KEY")")


st.set_page_config(
    page_title="PolicyPulse GDPR Assistant",
    layout="wide"
)

st.title("⚖️ PolicyPulse - GDPR RAG System")


@st.cache_resource
def process_data(pdf_path):

    loader = PyPDFLoader(pdf_path)

    documents = loader.load()

    splitter = TokenTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = splitter.split_documents(documents)

    for i, chunk in enumerate(chunks):

        chunk.metadata["chunk_id"] = i

        if "page" not in chunk.metadata:
            chunk.metadata["page"] = "Unknown"

    return chunks


@st.cache_resource
def build_vectorstore(_chunks):

    embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2"
    )

    vectorstore = Chroma.from_documents(
        documents=_chunks,
        embedding=embeddings,
        persist_directory="./chroma_db",
        collection_metadata={
            "hnsw:space": "cosine"
        }
    )

    return vectorstore


st.sidebar.header("📂 Upload GDPR PDF")

uploaded_file = st.sidebar.file_uploader(
    "Upload PDF File",
    type="pdf"
)


if uploaded_file is None:

    st.info("👈 Please upload the GDPR PDF from the sidebar.")

else:

    with open("temp_gdpr.pdf", "wb") as f:
        f.write(uploaded_file.getbuffer())

    with st.spinner("Processing GDPR document..."):

        chunks = process_data("temp_gdpr.pdf")

        vectorstore = build_vectorstore(chunks)

    st.success("✅ GDPR Knowledge Base Ready!")

    query = st.text_input(
        "Ask a GDPR Question"
    )

    ask_button = st.button("Generate Answer")

    if ask_button and query:

        # Check API key
        if not os.getenv("GROQ_API_KEY"):
            st.error(
                "❌ GROQ_API_KEY not found. Please add it to your .env file."
            )
            st.stop()

        llm = ChatGroq(
            groq_api_key=os.getenv("GROQ_API_KEY"),
            model_name="llama-3.3-70b-versatile",
            temperature=0
        )

        docs_and_scores = vectorstore.similarity_search_with_score(
            query,
            k=5
        )

        processed_results = []

        for doc, distance in docs_and_scores:

            cosine_similarity = 1 - distance

            processed_results.append(
                (doc, cosine_similarity)
            )

        processed_results.sort(
            key=lambda x: x[1],
            reverse=True
        )

        SIMILARITY_THRESHOLD = 0.5

        filtered_results = [
            (doc, score)
            for doc, score in processed_results
            if score >= SIMILARITY_THRESHOLD
        ]

        if not filtered_results:

            st.warning(
                "⚠️ No strongly relevant GDPR text found."
            )

            st.stop()

        retrieved_docs = [
            doc for doc, score in filtered_results
        ]

        context = "\n\n".join(
            [doc.page_content for doc in retrieved_docs]
        )

        template = """
You are a GDPR legal assistant.

Answer ONLY using the provided context.

Rules:
- Cite article numbers clearly.
- Only cite article numbers explicitly mentioned in the context.
- Do NOT invent citations.
- If information is missing, say:
  "Not found in retrieved GDPR text."

Context:
{context}

Question:
{question}

Answer:
"""

        prompt = PromptTemplate.from_template(template)

        rag_chain = (
            prompt
            | llm
            | StrOutputParser()
        )

        with st.spinner("Generating answer..."):

            answer = rag_chain.invoke({
                "context": context,
                "question": query
            })

        fact_template = """
You are a strict GDPR fact checker.

Check whether the answer is FULLY supported by the context.

Context:
{context}

Answer:
{answer}

If fully supported reply exactly:
VERIFIED

Otherwise reply exactly:
WARNING
"""

        fact_prompt = PromptTemplate.from_template(
            fact_template
        )

        fact_chain = (
            fact_prompt
            | llm
            | StrOutputParser()
        )

        verification = fact_chain.invoke({
            "context": context,
            "answer": answer
        })

        st.subheader("💡 Final Answer")

        st.write(answer)

        st.subheader("🔎 Fact Check Status")

        if "VERIFIED" in verification:

            st.success(
                "✅ The answer is fully supported by retrieved GDPR text."
            )

        else:

            st.error(
                "❌ WARNING: Possible hallucinated information detected."
            )

        st.subheader("📄 Retrieved Chunks")

        for i, (doc, score) in enumerate(filtered_results):

            chunk_id = doc.metadata.get(
                "chunk_id",
                "Unknown"
            )

            page_num = doc.metadata.get(
                "page",
                "Unknown"
            )

            st.markdown(f"## Chunk {i+1}")

            st.write(f"**Chunk ID:** {chunk_id}")

            st.write(f"**Page Number:** {page_num}")

            st.write(
                f"**Cosine Similarity:** {round(score, 4)}"
            )

            st.write("""
Retriever selected this chunk because its embedding
was semantically similar to the user query.
Results are ranked from highest to lowest cosine similarity.
""")

            st.code(doc.page_content)

Overwriting app.py


In [10]:
!streamlit run app.py &>/content/logs.txt &
!npm install localtunnel
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧
up to date, audited 23 packages in 2s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠇⠙⠹your url is: https://fuzzy-planes-design.loca.lt
^C
